# Eksperimen_Gevinta — RiskTrace: Deteksi Anomali Transaksi E-Wallet

**Nama:** Gevinta  
**Dataset:** risktrace_raw.csv  
**Target:** `is_suspicious` (0 = Normal, 1 = Suspicious)  
**Tujuan:** Melakukan EDA mendalam dan preprocessing data transaksi e-wallet untuk membangun sistem deteksi anomali.

---

## Section 1: Import Library & Load Data

In [ ]:
# ── Import Library ─────────────────────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# Styling
plt.rcParams['figure.dpi']     = 120
plt.rcParams['font.family']    = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('husl')

print('Library berhasil diimpor!')
print(f'  pandas     : {pd.__version__}')
print(f'  numpy      : {np.__version__}')
print(f'  seaborn    : {sns.__version__}')

In [ ]:
# ── Load Data ──────────────────────────────────────────────────────────────────
DATA_PATH = os.path.join('..', 'risktrace_raw.csv')
df = pd.read_csv(DATA_PATH)

print(f'Dataset dimuat: {DATA_PATH}')
print(f'Shape         : {df.shape}')
print(f'Kolom         : {df.columns.tolist()}')
df.head()

In [ ]:
# ── Informasi Umum Dataset ─────────────────────────────────────────────────────
print('=' * 50)
print('INFO DATASET')
print('=' * 50)
df.info()
print('\n')
df.describe(include='all').T

## Section 2: Exploratory Data Analysis (EDA)

### 2.1 Distribusi Target `is_suspicious`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
counts = df['is_suspicious'].value_counts()
colors = ['#4CAF50', '#F44336']
bars = axes[0].bar(
    ['Normal (0)', 'Suspicious (1)'],
    counts.values,
    color=colors, width=0.5, edgecolor='white', linewidth=1.5
)
for bar, count in zip(bars, counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 100,
        f'{count:,}\n({count/len(df)*100:.1f}%)',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
axes[0].set_title('Distribusi Target is_suspicious', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Transaksi')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Pie chart
axes[1].pie(
    counts.values,
    labels=['Normal (0)', 'Suspicious (1)'],
    colors=colors,
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title('Proporsi Kelas Target', fontsize=13, fontweight='bold')

plt.suptitle('Analisis Distribusi Kelas Target — RiskTrace', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('target_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'\nDistribusi target:\n{counts}')
print(f'\nRasio imbalance: {counts[1]/counts[0]:.2f}x lebih banyak Suspicious')

### 2.2 Statistik per Profile

In [ ]:
# Statistik per profile
profile_stats = df.groupby('profile').agg(
    jumlah_transaksi=('is_suspicious', 'count'),
    suspicious_count=('is_suspicious', 'sum'),
    suspicious_rate=('is_suspicious', 'mean'),
    avg_amount=('amount', 'mean'),
    avg_burst_score=('burst_score', 'mean'),
    avg_tx_count_7d=('tx_count_7d', 'mean'),
).round(4)

print('Statistik per Profile:')
display(profile_stats)

In [ ]:
# Visualisasi: suspicious rate per profile
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

profile_order = ['normal', 'early_stage', 'escalating', 'high_frequency_user']
palette = {'normal': '#4CAF50', 'early_stage': '#FFC107', 'escalating': '#FF5722', 'high_frequency_user': '#F44336'}

# Suspicious rate per profile
rates = profile_stats['suspicious_rate'].reindex(profile_order)
bars = axes[0].bar(
    profile_order, rates.values * 100,
    color=[palette[p] for p in profile_order],
    edgecolor='white', linewidth=1.5
)
for bar, rate in zip(bars, rates.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f'{rate*100:.1f}%', ha='center', va='bottom', fontweight='bold'
    )
axes[0].set_title('Suspicious Rate per Profile', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Suspicious Rate (%)')
axes[0].set_xlabel('Profile')

# Jumlah transaksi per profile
counts_p = df['profile'].value_counts().reindex(profile_order)
axes[1].bar(
    profile_order, counts_p.values,
    color=[palette[p] for p in profile_order],
    edgecolor='white', linewidth=1.5
)
for i, v in enumerate(counts_p.values):
    axes[1].text(i, v + 100, f'{v:,}', ha='center', va='bottom', fontweight='bold')
axes[1].set_title('Jumlah Transaksi per Profile', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Jumlah Transaksi')
axes[1].set_xlabel('Profile')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Analisis Profile Pengguna — RiskTrace', fontsize=14)
plt.tight_layout()
plt.savefig('profile_analysis.png', bbox_inches='tight', dpi=150)
plt.show()

### 2.3 Korelasi Fitur vs Target

In [ ]:
# Hitung korelasi numerik dengan target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = (
    df[numeric_cols].corr()['is_suspicious']
    .drop('is_suspicious')
    .abs()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 7))
colors_corr = ['#E53935' if v > 0.1 else '#42A5F5' for v in corr_with_target.values]
bars = ax.barh(
    corr_with_target.index,
    corr_with_target.values,
    color=colors_corr, edgecolor='white'
)
ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
ax.set_xlabel('|Korelasi| dengan is_suspicious', fontsize=11)
ax.set_title('Korelasi Absolut Fitur vs Target (is_suspicious)', fontsize=13, fontweight='bold')
ax.axvline(0.1, color='orange', linestyle='--', linewidth=1.5, label='Threshold 0.1')
ax.legend()
plt.tight_layout()
plt.savefig('feature_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nTop 5 fitur berkorelasi dengan target:')
print(corr_with_target.head(5).to_string())

In [ ]:
# Heatmap korelasi antar fitur numerik (top 10)
top_features = corr_with_target.head(10).index.tolist() + ['is_suspicious']

fig, ax = plt.subplots(figsize=(11, 9))
corr_matrix = df[top_features].corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, linewidths=0.5,
    cbar_kws={'shrink': 0.8},
    ax=ax
)
ax.set_title('Heatmap Korelasi — Top 10 Fitur + Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Visualisasi distribusi 4 fitur numerik penting per kelas
key_features = ['burst_score', 'tx_count_7d', 'amount', 'night_ratio_7d']
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for label, color, name in [(0, '#4CAF50', 'Normal'), (1, '#F44336', 'Suspicious')]:
        axes[i].hist(
            df[df['is_suspicious'] == label][feat],
            bins=40, alpha=0.6, color=color, label=name, edgecolor='white', linewidth=0.3
        )
    axes[i].set_title(f'Distribusi: {feat}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frekuensi')
    axes[i].legend()

plt.suptitle('Distribusi Fitur Kunci per Kelas — RiskTrace', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 3: Data Preprocessing

### 3.1 Handle Missing Values

In [ ]:
print('Missing values per kolom:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else '✅ Tidak ada missing values!')

print(f'\nJumlah duplikat: {df.duplicated().sum()}')
print(f'Shape awal     : {df.shape}')

### 3.2 Encoding Categorical Features

In [ ]:
# Drop kolom non-fitur
df_proc = df.drop(columns=['account_id', 'is_gambling'], errors='ignore').copy()
print(f'Shape setelah drop kolom ID: {df_proc.shape}')

# Label Encoding
le = LabelEncoder()
categorical_cols = ['profile', 'channel']
encoding_map = {}

for col in categorical_cols:
    if col in df_proc.columns:
        le.fit(df_proc[col].astype(str))
        encoding_map[col] = dict(zip(le.classes_, le.transform(le.classes_)))
        df_proc[col] = le.transform(df_proc[col].astype(str))
        print(f'  Label Encoded [{col}]: {encoding_map[col]}')

df_proc.head()

### 3.3 Feature Scaling

In [ ]:
TARGET_COL = 'is_suspicious'
X = df_proc.drop(columns=[TARGET_COL])
y = df_proc[TARGET_COL]

# Standard Scaling
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=X.columns
)

print(f'X shape setelah scaling: {X_scaled.shape}')
print(f'y distribusi            : {dict(y.value_counts())}')
X_scaled.describe().T[['mean', 'std', 'min', 'max']].round(3)

### 3.4 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'y_train distribusi: {dict(y_train.value_counts())}')
print(f'y_test  distribusi: {dict(y_test.value_counts())}')

### 3.5 Handle Class Imbalance dengan SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f'Sebelum SMOTE — y_train: {dict(y_train.value_counts())}')
print(f'Setelah SMOTE — y_train: {dict(pd.Series(y_train_res).value_counts())}')
print(f'X_train shape setelah SMOTE: {X_train_res.shape}')

# Visualisasi before/after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, y_plot, title in [
    (axes[0], y_train,     'Sebelum SMOTE'),
    (axes[1], pd.Series(y_train_res), 'Setelah SMOTE'),
]:
    counts_s = y_plot.value_counts()
    ax.bar(['Normal (0)', 'Suspicious (1)'], counts_s.values,
           color=['#4CAF50', '#F44336'], edgecolor='white')
    for i, v in enumerate(counts_s.values):
        ax.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Jumlah Sampel')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.suptitle('Efek SMOTE pada Training Set', fontsize=13)
plt.tight_layout()
plt.savefig('smote_effect.png', bbox_inches='tight', dpi=150)
plt.show()

### 3.6 Simpan Output Preprocessing

In [ ]:
# Gabungkan semua data (train + test, sudah di-SMOTE) untuk disimpan
X_all = pd.concat([
    pd.DataFrame(X_train_res, columns=X.columns),
    pd.DataFrame(X_test.values, columns=X.columns)
], ignore_index=True)

y_all = pd.concat([
    pd.Series(y_train_res, name=TARGET_COL),
    y_test.reset_index(drop=True)
], ignore_index=True)

df_final = X_all.copy()
df_final[TARGET_COL] = y_all

out_path = 'risktrace_preprocessing.csv'
df_final.to_csv(out_path, index=False)
print(f'✅ Preprocessing selesai!')
print(f'   Output: {out_path}')
print(f'   Shape : {df_final.shape}')
df_final.head()

## Section 4: Kesimpulan & Next Steps

### Temuan Utama EDA

| Aspek | Temuan |
|-------|--------|
| **Distribusi Target** | Dataset memiliki imbalance: 61.2% Suspicious vs 38.8% Normal |
| **Profil Paling Berisiko** | `escalating` dan `high_frequency_user` memiliki suspicious rate tertinggi |
| **Fitur Terpenting** | `burst_score`, `tx_count_7d`, `drain_cycle_flag`, `night_ratio_7d` |
| **Pola Transaksi** | Transaksi malam hari (`is_night=1`) dan burst aktivitas tinggi (`burst_score > 0.7`) sangat berkorelasi dengan suspicious |
| **Round Amount** | `round_amount_flag` (amount kelipatan 50.000) sering muncul pada transaksi suspicious |

### Preprocessing yang Dilakukan

1. ✅ **Drop kolom non-fitur**: `account_id`, `is_gambling`
2. ✅ **Label Encoding**: `profile`, `channel`
3. ✅ **Standard Scaling**: seluruh fitur numerik (mean=0, std=1)
4. ✅ **Train-Test Split**: 80/20, stratified
5. ✅ **SMOTE**: training set di-balance menjadi 50/50

### Next Steps — Modelling

- Train **Logistic Regression** (baseline)
- Train **XGBoost** (model utama) dengan MLflow autolog
- Train **Isolation Forest** (anomaly detection)
- **Hyperparameter tuning** XGBoost dengan GridSearchCV
- Tracking eksperimen dengan **MLflow + DagsHub**
- Monitoring model dengan **Prometheus + Grafana**